# Build tables of passport data for harvest by GOOS

This notebook walks through constructing two tables: one for platform passports, one for sensors, which GOOS will harvest.

**This assumes you have your datasets on an ERDDAP server and will require some customisation if those datasets are not in OG1 format**

In [1]:
import json
from pathlib import Path
import requests
import xarray as xr
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
from erddapy import ERDDAP

### Authentication

The GOOS API requires authentication, see the [API token guide](https://www.ocean-ops.org/passports/#/tokens)

You can add your auth in a file `secrets.json` in the home directory of this repo, or directly in the cell below

In [2]:
oceanops_auth = {'X-OceanOPS-Metadata-ID': 'nnnn',
 'X-OceanOPS-Metadata-Token': '16-char-secret-token'}

if Path('../secrets.json').exists():
    with open('../secrets.json') as f:
        oceanops_auth = json.load(f)
else:
    print("enter your oceanOPS authentication in the dictionary `oceanops_auth` in this cell")

In [3]:
headers = {
    'Content-Type': 'application/json' ,
    'accept': 'application/json' ,
    **oceanops_auth
}
output_dir = Path('output')
if not output_dir.exists():
    output_dir.mkdir()

In [4]:
wmo_id = 6801674
platform_serial = 'SEA055'
df_oceanops = pd.read_csv(output_dir / f'goos_data_{wmo_id}.csv')
for col in df_oceanops.columns:
    if 'date' in col.lower():
        df_oceanops[col] = pd.to_datetime(df_oceanops[col])

In [5]:
e = ERDDAP(
    server="http://136.243.54.252:8082/erddap",
    protocol="tabledap",
)
e.response = "csv"
e.dataset_id = "allDatasets"
df_datasets = e.to_pandas()
df_datasets = df_datasets[df_datasets.datasetID.str.contains("_R")] # these are our nrt OG1 datasets
df_ids = df_datasets.datasetID.str.split('_',expand=True)
df_datasets['platform_serial'] = df_ids[0]
df_datasets['mission'] = df_ids[1]
df_platform = df_datasets[df_datasets.platform_serial==platform_serial.lower()]
df_platform = df_platform.rename({'minTime (UTC)': 'deployment_date', 'maxTime (UTC)': 'recovery_date'}, axis=1)
subset_vars =['datasetID', 'title',  'summary', 'platform_serial', 'mission', 'deployment_date', 'recovery_date']
df_platform['deployment_date'] = pd.to_datetime(df_platform.deployment_date)
df_platform['recovery_date'] = pd.to_datetime(df_platform.recovery_date)
df_platform = df_platform[subset_vars]
df_platform = df_platform.sort_values('deployment_date')
df_platform['gtsId'] = wmo_id
print(f"Found data for {len(df_platform)} deployments on ERDDAP")

Found data for 33 deployments on ERDDAP


In [6]:
df_platform['wigosId'] = ''
matches = 0
for i, row in df_platform.iterrows():
    deploy_diff = df_oceanops.deployment_date - row.deployment_date
    time_match = np.abs(deploy_diff) < np.timedelta64(3, 'D')
    if time_match.any():
        matching_passports = df_oceanops[time_match]
        if len(matching_passports == 1):
            matches += 1
            df_platform.loc[i, 'wigosId'] = matching_passports['wigosId'].values[0]
        else:
            print("Error! More than one match detected")
print(f'Matched {matches} datasets')

dfa = df_platform[['datasetID', 'mission', 'title', 'deployment_date', 'recovery_date', 'wigosId', 'gtsId', 'platform_serial']]

dfb =df_oceanops[['passport_num', 'wigosId', 'name','deployment_date', 'identification_internalId', 'retrieval_startDate','retrieval_endDate', 'retrieval_endingCause' ]]
df = pd.merge(dfa, dfb, left_on='wigosId', right_on='wigosId', suffixes=('_voto', '_ops'), how='left')
df[df.wigosId!= ''].head()

Matched 33 datasets


,datasetID,mission,title,deployment_date_voto,recovery_date,wigosId,gtsId,platform_serial,passport_num,name,deployment_date_ops,identification_internalId,retrieval_startDate,retrieval_endDate,retrieval_endingCause
0,sea055_20201031T0902_R,20201031T0902,nrt_SEA055_M16 SEA055_M16 Kaprifol,2020-10-31 09:03:00+00:00,2020-11-19 10:00:30+00:00,0-22000-0-6801674,6801674,sea055,0,Kaprifol,2020-10-31 09:03:00+00:00,Kaprifol55-20201031T0902,2020-11-19 10:00:30+00:00,2020-11-19 10:00:30+00:00,"{'code': 'voluntary-recovery-of-the-platform',..."
1,sea055_20210305T0812_R,20210305T0812,nrt_SEA055_M18 SEA055_M18 Kaprifol,2021-03-05 08:13:31+00:00,2021-03-25 11:39:00+00:00,0-22000-1-6801674,6801674,sea055,1,Kaprifol,2021-03-05 08:13:31+00:00,Kaprifol55-20210305T0813,2021-03-25 11:39:00+00:00,2021-03-25 11:39:00+00:00,"{'code': 'voluntary-recovery-of-the-platform',..."
2,sea055_20210326T0937_R,20210326T0937,nrt_SEA055_M19 SEA055_M19 Kaprifol,2021-03-26 09:38:30+00:00,2021-04-14 04:46:00+00:00,0-22000-2-6801674,6801674,sea055,2,Kaprifol,2021-03-26 09:38:30+00:00,Kaprifol55-20210326T0938,2021-04-14 04:46:00+00:00,2021-04-14 04:46:00+00:00,"{'code': 'voluntary-recovery-of-the-platform',..."
3,sea055_20210414T1704_R,20210414T1704,nrt_SEA055_M20 SEA055_M20 Kaprifol,2021-04-14 17:13:03+00:00,2021-05-02 16:11:30+00:00,0-22000-3-6801674,6801674,sea055,3,Kaprifol,2021-04-14 17:13:03+00:00,Kaprifol55-20210414T1713,2021-05-02 16:11:30+00:00,2021-05-02 16:11:30+00:00,"{'code': 'voluntary-recovery-of-the-platform',..."
4,sea055_20210503T0601_R,20210503T0601,nrt_SEA055_M21 SEA055_M21 Kaprifol,2021-05-03 06:19:00+00:00,2021-05-21 14:57:00+00:00,0-22000-4-6801674,6801674,sea055,4,Kaprifol,2021-05-03 06:19:00+00:00,Kaprifol55-20210503T0609,2021-05-21 14:57:00+00:00,2021-05-21 14:57:00+00:00,"{'code': 'voluntary-recovery-of-the-platform',..."


# Generate tables for ERDDAP that oceanOPS will harvest

This is an example pulling from the VOTO ERDDAP. ymmv!

Using the CSV column names from https://www.ocean-ops.org/passports/#/submit

We work through the WIGOS Passport ids that we have, extract metadata from the ERDDAP dataset of each mission, and compile these metadata into two tables, following the example from GOOS.

Note that the network comes from this table https://www.ocean-ops.org/passports/#/code-tables?code=CT-GNW

In [7]:
passport_dicts = []
sensor_dicts = []
for i, row in df.iterrows():
    dataset_id = row.datasetID
    e.dataset_id = dataset_id
    passport_id = row.wigosId
    platform_serial = row.platform_serial
    
    ds = e.to_xarray()

    attrs = ds.attrs
    
    platform_serial = dataset_id.split('_M')[0]
    passport_dict = {'PASSPORT IDENTIFIER': passport_id,
                     'PROGRAM': 'VOTO: Ocean Knowledge glider program',
                     'PLATFORM MODEL': 'SeaExplorer X2',
                     'PLATFORM SERIAL NUMBER': platform_serial,
                     'DEPLOYMENT DATE': str(ds.time.values[~np.isnan(ds.time)][0])[:19] + 'Z',
                     'DEPLOYMENT LATITUDE': ds.latitude.values[~np.isnan(ds.latitude)][0],
                     'DEPLOYMENT LONGITUDE': ds.longitude.values[~np.isnan(ds.longitude)][0],
                     'DEPLOYMENT SHIP': 'Seeker', # tbd
                     'GTS ID': int(attrs['wmo_id']),
                     'TELECOM TYPE': 'iridium',                    
    }
    
    # conditional entries
    conditional_dict = {
        'PRINCIPAL INVESTIGATOR': 'Louise Biddle',
        'AGENCIES': 'VOTO',
        'AGENCY_ROLES': 'operating agency',
        'CONTRIBUTORS': attrs['contributor_name'],
        'CONTRIBUTOR_ROLE': attrs['contributor_role']
        
    }
    end_time = ds.time.values[~np.isnan(ds.time)][-1]
    
    if end_time < np.datetime64(datetime.datetime.now() - datetime.timedelta(days=3)):
        passport_dict['END DATE'] = str(ds.time.values[~np.isnan(ds.time)][-1])[:19] + 'Z'
        passport_dict['ENDING CAUSE'] = 'recovered'
    passport_dict = passport_dict | conditional_dict
    passport_dicts.append(passport_dict)  

    from string import ascii_uppercase as auc
    i = 0
    for var_name in ds.variables:
        variable = ds[var_name]
        attrs = variable.attrs
        if 'sensor' not in attrs.keys():
            continue
        if attrs['sensor'][:6] != 'SENSOR':
            continue
        sensor_dict = ds[attrs['sensor']].attrs
        if 'standard_name' not in attrs.keys():
            continue
        sensor_dict = {
            'PASSPORT IDENTIFIER': passport_id,
            'SHORTCODE': auc[i],
            'SENSOR MODEL': sensor_dict['sensor_model'],
            'VARIABLE': attrs['standard_name'],
            'SERIAL NUMBER': sensor_dict['sensor_serial_number'],
        }
        i += 1
        sensor_dicts.append(sensor_dict)
        

In [8]:
df_passport = pd.DataFrame(passport_dicts)
df_passport.to_csv('passport.csv', sep=';', index=False)
df_passport.head()

,PASSPORT IDENTIFIER,PROGRAM,PLATFORM MODEL,PLATFORM SERIAL NUMBER,DEPLOYMENT DATE,DEPLOYMENT LATITUDE,DEPLOYMENT LONGITUDE,DEPLOYMENT SHIP,GTS ID,TELECOM TYPE,END DATE,ENDING CAUSE,PRINCIPAL INVESTIGATOR,AGENCIES,AGENCY_ROLES,CONTRIBUTORS,CONTRIBUTOR_ROLE
0,0-22000-0-6801674,VOTO: Ocean Knowledge glider program,SeaExplorer X2,sea055_20201031T0902_R,2020-10-31T09:03:00Z,55.565055,15.693231,Seeker,6801674,iridium,2020-11-19T10:00:30Z,recovered,Louise Biddle,VOTO,operating agency,"Callum Rollo, Louise Biddle,, , , , ,","Data scientist, PI, Operator, Operator, Operat..."
1,0-22000-1-6801674,VOTO: Ocean Knowledge glider program,SeaExplorer X2,sea055_20210305T0812_R,2021-03-05T08:13:31Z,55.570673,16.365672,Seeker,6801674,iridium,2021-03-25T11:39:00Z,recovered,Louise Biddle,VOTO,operating agency,"Callum Rollo, Louise Biddle,, , , , ,","Data scientist, PI, Operator, Operator, Operat..."
2,0-22000-2-6801674,VOTO: Ocean Knowledge glider program,SeaExplorer X2,sea055_20210326T0937_R,2021-03-26T09:38:30Z,55.570853,16.351618,Seeker,6801674,iridium,2021-04-14T04:46:00Z,recovered,Louise Biddle,VOTO,operating agency,"Callum Rollo, Louise Biddle,, , , , ,","Data scientist, PI, Operator, Operator, Operat..."
3,0-22000-3-6801674,VOTO: Ocean Knowledge glider program,SeaExplorer X2,sea055_20210414T1704_R,2021-04-14T17:13:03Z,55.570852,16.350296,Seeker,6801674,iridium,2021-05-02T16:11:30Z,recovered,Louise Biddle,VOTO,operating agency,"Callum Rollo, Louise Biddle,, , , , ,","Data scientist, PI, Operator, Operator, Operat..."
4,0-22000-4-6801674,VOTO: Ocean Knowledge glider program,SeaExplorer X2,sea055_20210503T0601_R,2021-05-03T06:19:00Z,55.571255,16.351489,Seeker,6801674,iridium,2021-05-21T14:57:00Z,recovered,Louise Biddle,VOTO,operating agency,"Callum Rollo, Louise Biddle,, , , , ,","Data scientist, PI, Operator, Operator, Operat..."


In [9]:
df_sensor = pd.DataFrame(sensor_dicts)
df_sensor.to_csv('sensor.csv', sep=';', index=False)

df_sensor.head()

,PASSPORT IDENTIFIER,SHORTCODE,SENSOR MODEL,VARIABLE,SERIAL NUMBER
0,0-22000-0-6801674,A,WET Labs {Sea-Bird WETLabs} ECO Puck Triplet F...,acoustic_volume_backscattering_strength_in_sea...,5925
1,0-22000-0-6801674,B,WET Labs {Sea-Bird WETLabs} ECO Puck Triplet F...,concentration_of_colored_dissolved_organic_mat...,5925
2,0-22000-0-6801674,C,WET Labs {Sea-Bird WETLabs} ECO Puck Triplet F...,concentration_of_chlorophyll_in_sea_water,5925
3,0-22000-0-6801674,D,JFE Advantech Rinko FT ARO-FT oxygen sensor,mole_concentration_of_dissolved_molecular_oxyg...,0040
4,0-22000-0-6801674,E,Satlantic {Sea-Bird} OCR-504 multispectral rad...,downwelling_spherical_irradiance_in_sea_water,2043


In [10]:
df_oceanops.head()

,passport_num,id,wigosId,name,comments,deployment_date,deployment_latitude,deployment_longitude,deployment_ship,retrieval_startDate,retrieval_endDate,retrieval_endingCause,identification_internalId,telecom_type,hardware_mainDevice,affiliation_program,affiliation_networks
0,0,1145835,0-22000-0-6801674,Kaprifol,"raw, uncorrected salinity<br/>This study used ...",2020-10-31 09:03:00+00:00,55.565430,15.693250,"{'imoNumber': '7012179', 'ovhId': '1015198'}",2020-11-19 10:00:30+00:00,2020-11-19 10:00:30+00:00,"{'code': 'voluntary-recovery-of-the-platform',...",Kaprifol55-20201031T0902,{'code': 'iridium'},"{'model': {'code': 'seaexplorer'}, 'serialNumb...",{'code': 'voto-ocean-knowledge-glider-program'},"['boon', 'eoos', 'marginal-seas', 'oceanglider..."
1,1,1145836,0-22000-1-6801674,Kaprifol,"raw, uncorrected salinity<br/>This study used ...",2021-03-05 08:13:31+00:00,55.570671,16.365665,"{'imoNumber': '7012179', 'ovhId': '1015198'}",2021-03-25 11:39:00+00:00,2021-03-25 11:39:00+00:00,"{'code': 'voluntary-recovery-of-the-platform',...",Kaprifol55-20210305T0813,{'code': 'iridium'},"{'model': {'code': 'seaexplorer'}, 'serialNumb...",{'code': 'voto-ocean-knowledge-glider-program'},"['boon', 'eoos', 'marginal-seas', 'oceanglider..."
2,2,1145837,0-22000-2-6801674,Kaprifol,"raw, uncorrected salinity<br/>This study used ...",2021-03-26 09:38:30+00:00,55.570671,16.351534,"{'imoNumber': '7012179', 'ovhId': '1015198'}",2021-04-14 04:46:00+00:00,2021-04-14 04:46:00+00:00,"{'code': 'voluntary-recovery-of-the-platform',...",Kaprifol55-20210326T0938,{'code': 'iridium'},"{'model': {'code': 'seaexplorer'}, 'serialNumb...",{'code': 'voto-ocean-knowledge-glider-program'},"['boon', 'eoos', 'marginal-seas', 'oceanglider..."
3,3,1145838,0-22000-3-6801674,Kaprifol,"raw, uncorrected salinity<br/>This study used ...",2021-04-14 17:13:03+00:00,55.570751,16.350166,"{'imoNumber': '7012179', 'ovhId': '1015198'}",2021-05-02 16:11:30+00:00,2021-05-02 16:11:30+00:00,"{'code': 'voluntary-recovery-of-the-platform',...",Kaprifol55-20210414T1713,{'code': 'iridium'},"{'model': {'code': 'seaexplorer'}, 'serialNumb...",{'code': 'voto-ocean-knowledge-glider-program'},"['boon', 'eoos', 'marginal-seas', 'oceanglider..."
4,4,1145839,0-22000-4-6801674,Kaprifol,"raw, uncorrected salinity<br/>This study used ...",2021-05-03 06:19:00+00:00,55.571247,16.351551,"{'imoNumber': '7012179', 'ovhId': '1015198'}",2021-05-21 14:57:00+00:00,2021-05-21 14:57:00+00:00,"{'code': 'voluntary-recovery-of-the-platform',...",Kaprifol55-20210503T0609,{'code': 'iridium'},"{'model': {'code': 'seaexplorer'}, 'serialNumb...",{'code': 'voto-ocean-knowledge-glider-program'},"['boon', 'eoos', 'marginal-seas', 'oceanglider..."


These tables can then be hosted on ERDDAP for OceanOps/GOOS to harvest. The datasets for these examples are live at:

- https://erddap.observations.voiceoftheocean.org/erddap/tabledap/goos_passport_dataset.html
- https://erddap.observations.voiceoftheocean.org/erddap/tabledap/goos_sensor_dataset.html